In [ ]:
wsl

In [ ]:
#로컬 CPU 개수 확인
nproc

In [ ]:
conda activate docking

In [ ]:
cat config.txt

# 트러블 슈팅

In [ ]:
#발생한 에러
Process Process-6:
Process Process-5:
Traceback (most recent call last):
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/c/Users/PC/OneDrive - emocog/문서/Simoa_Jay/8)깃허브/LAIDD_drug_discovery/structure_based_drug_discovery/3_molecular_docking/3_4_multi_processing_for_virtual_screening/mdock_vina.py", line 96, in worker
    ligandtools.pdbqt_to_pdb_ref(dock_pdbqt_file, dock_pdb_file, pdb_file)
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/site-packages/pbi/ligandtools.py", line 359, in pdbqt_to_pdb_ref
    model_dict = read_pdbqt_file(input_pdbqt_file)
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/site-packages/pbi/ligandtools.py", line 291, in read_pdbqt_file
    fp = open(pdbqt_file)
FileNotFoundError: [Errno 2] No such file or directory: 'dock_mp/CHEMBL5/dock_CHEMBL517068.pdbqt'
Traceback (most recent call last):
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/c/Users/PC/OneDrive - emocog/문서/Simoa_Jay/8)깃허브/LAIDD_drug_discovery/structure_based_drug_discovery/3_molecular_docking/3_4_multi_processing_for_virtual_screening/mdock_vina.py", line 96, in worker
    ligandtools.pdbqt_to_pdb_ref(dock_pdbqt_file, dock_pdb_file, pdb_file)
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/site-packages/pbi/ligandtools.py", line 359, in pdbqt_to_pdb_ref
    model_dict = read_pdbqt_file(input_pdbqt_file)
  File "/home/kucz11/miniconda310/envs/docking/lib/python3.10/site-packages/pbi/ligandtools.py", line 291, in read_pdbqt_file
    fp = open(pdbqt_file)
FileNotFoundError: [Errno 2] No such file or directory: 'dock_mp/CHEMBL3/dock_CHEMBL341280.pdbqt'
Process Process-4:

In [ ]:
#트러블 슈팅
# 1. gen_3d 테스트
~/miniconda310/envs/docking/bin/python -c "
from pbi import ligandtools
ligandtools.gen_3d('Cc1cccc(n1)c1nc(c2ccccc2n1)Nc1ccncc1', '/tmp/test.pdb', mol_id='test', file_format='pdb')
print('gen_3d OK')
"

# 2. pdb→pdbqt 테스트
~/miniconda310/envs/docking/bin/python -c "
from pbi.pdbtools import PDBtools
PDBtools.ligand_to_pdbqt('/tmp/test.pdb', '/tmp/test.pdbqt')
print('pdbqt OK')
"

# 3. qvina02 테스트
~/docking_tools/qvina02 --config config.txt --ligand /tmp/test.pdbqt --out /tmp/dock_test.pdbqt



In [ ]:
# 2단계 결과 확인
ls -la /tmp/test.pdb /tmp/test.pdbqt 2>&1

# pdbqt 변환 오류 직접 확인
~/miniconda310/envs/docking/bin/python -c "
from pbi.pdbtools import PDBtools
e = PDBtools.ligand_to_pdbqt('/tmp/test.pdb', '/tmp/test.pdbqt')
print('error:', e)
" 2>&1

In [ ]:
/home/kucz11/miniconda310/envs/docking/bin/prepare_ligand4 \
    -l /tmp/test.pdb \
    -o /tmp/test.pdbqt \
    -U nphs_lps 2>&1

In [ ]:
# obabel로 직접 pdb → pdbqt 변환 테스트
/home/kucz11/miniconda310/envs/docking/bin/obabel /tmp/test.pdb -o pdbqt -O /tmp/test.pdbqt --partialcharge gasteiger 2>&1
ls -la /tmp/test.pdbqt

In [ ]:
# 현재 ligand_to_pdbqt 함수 확인 (1209~1230번째 줄)
sed -n '1209,1232p' ~/miniconda310/envs/docking/lib/python3.10/site-packages/pbi/pdbtools.py

In [ ]:
notepad "C:\Users\PC\OneDrive - emocog\문서\Simoa_Jay\8)깃허브\LAIDD_drug_discovery\structure_based_drug_discovery\3_molecular_docking\3_4_multi_processing_for_virtual_screening\mdock_vina.py"

# 기존
ligandtools.pdbqt_to_pdb_ref(dock_pdbqt_file, dock_pdb_file, pdb_file)

# 수정
try:
    ligandtools.pdbqt_to_pdb_ref(dock_pdbqt_file, dock_pdb_file, pdb_file)
except Exception as e:
    print(f'pdbqt_to_pdb_ref warning: {e}', flush=True)

# 실행

In [ ]:
#필요시 기존 파일 삭제
rm -rf dock_mp docking_mp.txt

In [ ]:
#실행
~/miniconda310/envs/docking/bin/python mdock_vina.py \
    -v ~/docking_tools/qvina02 \
    -c config.txt \
    -s ligand_list.smi \
    -d dock_mp \
    -o docking_mp.txt \
    -p 4

In [ ]:
# 상위 10개 best score 확인
~/miniconda310/envs/docking/bin/python -c "
import csv

results = []
with open('docking_mp.txt') as f:
    next(f)  # 헤더 skip
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 3 and parts[2] != 'None':
            mol_id = parts[0]
            best_score = float(parts[2].split(',')[0])
            results.append((mol_id, best_score))

results.sort(key=lambda x: x[1])
print('Rank  mol_id           Best Score')
print('-' * 40)
for i, (mol_id, score) in enumerate(results[:10], 1):
    print(f'{i:>4}  {mol_id:<18} {score:.1f}')
"

# 결과 해석

## 멀티프로세싱 도킹 결과 (135개 리간드, 4코어 병렬)

### 상위 10개 Hit 화합물

| Rank | Mol ID | Best Score (kcal/mol) |
|------|--------|----------------------|
| 1 | **CHEMBL409356** | **-14.5** |
| 2 | CHEMBL260239 | -13.4 |
| 3 | CHEMBL406411 | -13.3 |
| 4 | CHEMBL567287 | -13.1 |
| 5 | CHEMBL436944 | -13.1 |
| 6 | CHEMBL585505 | -12.6 |
| 7 | CHEMBL260015 | -12.6 |
| 8 | CHEMBL204211 | -12.5 |
| 9 | CHEMBL206233 | -12.5 |
| 10 | CHEMBL566234 | -12.3 |

### 평가

- **CHEMBL409356 (-14.5 kcal/mol)** → 135개 중 가장 강한 결합 친화도
- 단일 도킹 Best Hit인 CHEMBL517068 (-10.5 kcal/mol)보다 **4.0 kcal/mol 더 강한 결합**
- 4코어 병렬 처리(`-p 4`)로 135개 리간드를 효율적으로 완료

> **Score 해석:** -14.5 kcal/mol은 매우 강한 결합 친화도  
> 일반적으로 -8 kcal/mol 이하면 strong binder로 간주